In [1]:
from google.colab import drive
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import os
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from peft import LoraConfig, TaskType, get_peft_model
drive.mount('/content/drive')
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CACHE_FILE = "/content/drive/MyDrive/dataMiningProject/CSI_Project/datasets/token_cache/tokens_maxlen512.pt"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
class SupervisedContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        features = F.normalize(features, p=2, dim=1)
        batch_size = features.shape[0]
        labels = labels.contiguous().view(-1, 1)
        mask = torch.eq(labels, labels.T).float().to(features.device)
        logits = torch.div(torch.matmul(features, features.T), self.temperature)
        logits_max, _ = torch.max(logits, dim=1, keepdim=True)
        logits = logits - logits_max.detach()
        logits_mask = torch.scatter(torch.ones_like(mask), 1, torch.arange(batch_size).view(-1, 1).to(features.device), 0)
        mask = mask * logits_mask
        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-6)
        mean_log_prob_pos = (mask * log_prob).sum(1) / (mask.sum(1) + 1e-6)
        return -mean_log_prob_pos.mean()

class RDropLoss(nn.Module):
    def __init__(self, alpha=4):
        super().__init__()
        self.alpha = alpha
        self.kl = nn.KLDivLoss(reduction='none')
    def forward(self, logits1, logits2, target, weights=None):
        ce = nn.CrossEntropyLoss(weight=weights)
        ce_loss = 0.5 * (ce(logits1, target) + ce(logits2, target))
        p_loss = self.kl(F.log_softmax(logits1, dim=-1), F.softmax(logits2, dim=-1))
        q_loss = self.kl(F.log_softmax(logits2, dim=-1), F.softmax(logits1, dim=-1))
        kl_loss = 0.5 * (p_loss.sum() + q_loss.sum()) / logits1.shape[0]
        return ce_loss + self.alpha * kl_loss

In [3]:
class VulnerabilityDataset(Dataset):
    def __init__(self, cache: dict, split: str):
        indices = [i for i, s in enumerate(cache["split_origins"]) if s == split]
        self.input_ids = cache["input_ids"][indices]
        self.attention_mask = cache["attention_mask"][indices]
        self.cwe_labels = cache["cwe_labels"][indices]
    def __len__(self): return len(self.input_ids)
    def __getitem__(self, idx):
        return {"input_ids": self.input_ids[idx], "attention_mask": self.attention_mask[idx], "cwe_label": self.cwe_labels[idx]}

if os.path.exists(CACHE_FILE):
    cache = torch.load(CACHE_FILE, map_location='cpu')
    train_ds = VulnerabilityDataset(cache, split="train")

    counts = np.bincount(train_ds.cwe_labels.numpy(), minlength=8).astype(float)
    CLASS_WEIGHTS = torch.tensor(len(train_ds) / (8 * counts), dtype=torch.float).to(DEVICE)

    sampler = WeightedRandomSampler(torch.tensor(1.0/(counts[train_ds.cwe_labels.numpy()]+1e-6)), len(train_ds))
    test_loader = DataLoader(train_ds, batch_size=8, sampler=sampler)

    print(f"✅ CLASS_WEIGHTS is now defined: {CLASS_WEIGHTS}")
else:
    print("❌ CACHE_FILE Path is incorrect!")

✅ CLASS_WEIGHTS is now defined: tensor([1.6762, 1.5166, 1.1727, 2.2159, 0.5737, 1.2194, 1.7693, 0.4326],
       device='cuda:0')


In [4]:
class GraphCodeBERTLoRACWEModel(nn.Module):
    def __init__(self, model_name="microsoft/graphcodebert-base", num_cwe_classes=8, class_weights=None):
        super().__init__()
        encoder = AutoModel.from_pretrained(model_name)
        lora_cfg = LoraConfig(task_type=TaskType.FEATURE_EXTRACTION, r=16, lora_alpha=32, target_modules=["query", "key", "value"])
        self.encoder = get_peft_model(encoder, lora_cfg)
        self.cwe_head = nn.Sequential(
            nn.LayerNorm(self.encoder.config.hidden_size),
            nn.Linear(self.encoder.config.hidden_size, self.encoder.config.hidden_size // 2),
            nn.GELU(),
            nn.Linear(self.encoder.config.hidden_size // 2, num_cwe_classes)
        )
        self.register_buffer("weights", class_weights)

    def forward(self, input_ids, attention_mask):
        enc = self.encoder(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
        features = enc.last_hidden_state[:, 0, :]
        logits = self.cwe_head(features)
        return {"logits": logits, "features": features}

#Test SCL LOSS & R-DROP LOSS on batch from real data
model = GraphCodeBERTLoRACWEModel(class_weights=CLASS_WEIGHTS).to(DEVICE)
scl_criterion = SupervisedContrastiveLoss().to(DEVICE)
rdrop_criterion = RDropLoss().to(DEVICE)

model.train()
batch = next(iter(test_loader))
ids, mask, labels = batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE), batch['cwe_label'].to(DEVICE)

# Double Pass
res1 = model(ids, mask)
res2 = model(ids, mask)

# الحساب
loss_scl = scl_criterion(torch.cat([res1['features'], res2['features']], dim=0), torch.cat([labels, labels], dim=0))
loss_rdrop = rdrop_criterion(res1['logits'], res2['logits'], labels, weights=CLASS_WEIGHTS)

print(f" SCL Loss: {loss_scl.item():.4f}")
print(f" R-Drop Loss: {loss_rdrop.item():.4f}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


 SCL Loss: 1.5855
 R-Drop Loss: 2.1594


In [ ]:
# Task 3: Integrate SCL into the training loop (SCL + R-Drop)
if "train_ds" not in globals():
    raise RuntimeError("train_ds is not defined. Run the dataset/cache cell first.")

# Reuse existing sampler if available; otherwise create a standard shuffled loader.
if "sampler" in globals():
    train_loader = DataLoader(train_ds, batch_size=8, sampler=sampler, drop_last=True)
else:
    train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, drop_last=True)

# Initialize model/losses if they do not already exist.
if "model" not in globals():
    model = GraphCodeBERTLoRACWEModel(class_weights=CLASS_WEIGHTS).to(DEVICE)
if "scl_criterion" not in globals():
    scl_criterion = SupervisedContrastiveLoss(temperature=0.07).to(DEVICE)
if "rdrop_criterion" not in globals():
    rdrop_criterion = RDropLoss(alpha=4).to(DEVICE)

num_epochs = 3
learning_rate = 2e-5
weight_decay = 0.01
scl_weight = 0.2

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
num_training_steps = len(train_loader) * num_epochs
num_warmup_steps = int(0.1 * num_training_steps)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    total_scl = 0.0
    total_rdrop = 0.0

    for batch in train_loader:
        ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        labels = batch["cwe_label"].to(DEVICE)

        optimizer.zero_grad()

        # Two stochastic forward passes for R-Drop, and shared features for SCL.
        out1 = model(ids, mask)
        out2 = model(ids, mask)

        loss_scl = scl_criterion(
            torch.cat([out1["features"], out2["features"]], dim=0),
            torch.cat([labels, labels], dim=0),
        )
        loss_rdrop = rdrop_criterion(out1["logits"], out2["logits"], labels, weights=CLASS_WEIGHTS)

        loss = loss_rdrop + scl_weight * loss_scl
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        total_scl += loss_scl.item()
        total_rdrop += loss_rdrop.item()

    n_batches = len(train_loader)
    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Total: {total_loss / n_batches:.4f} | "
        f"R-Drop: {total_rdrop / n_batches:.4f} | "
        f"SCL: {total_scl / n_batches:.4f}"
    )

In [ ]:
# Task 3+ : Training with T4 optimization + Checkpoint save/load
import torch.cuda.amp as amp
from pathlib import Path

# ============ T4 GPU Memory Optimizations ============
torch.cuda.empty_cache()
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

# ============ Checkpoint Management ============
CHECKPOINT_DIR = Path("/content/drive/MyDrive/dataMiningProject/CSI_Project/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LATEST_CHECKPOINT = CHECKPOINT_DIR / "latest.pt"

def save_checkpoint(epoch, model, optimizer, scheduler, loss_history):
    """Save training state to resume later."""
    checkpoint = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "loss_history": loss_history,
    }
    torch.save(checkpoint, LATEST_CHECKPOINT)
    print(f"✅ Checkpoint saved at epoch {epoch + 1}")

def load_checkpoint(model, optimizer, scheduler):
    """Load checkpoint if exists; return epoch and loss history."""
    if LATEST_CHECKPOINT.exists():
        checkpoint = torch.load(LATEST_CHECKPOINT, map_location=DEVICE)
        model.load_state_dict(checkpoint["model_state"])
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        scheduler.load_state_dict(checkpoint["scheduler_state"])
        loss_history = checkpoint["loss_history"]
        start_epoch = checkpoint["epoch"] + 1
        print(f"✅ Checkpoint loaded. Resuming from epoch {start_epoch + 1}")
        return start_epoch, loss_history
    return 0, {"loss": [], "scl": [], "rdrop": []}

# ============ Setup (T4-optimized) ============
if "train_ds" not in globals():
    raise RuntimeError("train_ds is not defined. Run the dataset/cache cell first.")

# T4-friendly batch size (8 is safe, can reduce to 4 if OOM)
BATCH_SIZE = 8
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler if "sampler" in globals() else None, shuffle=True if "sampler" not in globals() else False, drop_last=True)

if "model" not in globals():
    model = GraphCodeBERTLoRACWEModel(class_weights=CLASS_WEIGHTS).to(DEVICE)
if "scl_criterion" not in globals():
    scl_criterion = SupervisedContrastiveLoss(temperature=0.07).to(DEVICE)
if "rdrop_criterion" not in globals():
    rdrop_criterion = RDropLoss(alpha=4).to(DEVICE)

# ============ Training Config ============
num_epochs = 10
learning_rate = 2e-5
weight_decay = 0.01
scl_weight = 0.3
gradient_accumulation_steps = 2  # Effective batch = 8 * 2 = 16
use_mixed_precision = True  # T4 benefits from FP16

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
num_training_steps = (len(train_loader) // gradient_accumulation_steps) * num_epochs
num_warmup_steps = int(0.1 * num_training_steps)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

# Load checkpoint if available
start_epoch, loss_history = load_checkpoint(model, optimizer, scheduler)
scaler = amp.GradScaler()

# ============ Training Loop with Mixed Precision ============
for epoch in range(start_epoch, num_epochs):
    model.train()
    total_loss = 0.0
    total_scl = 0.0
    total_rdrop = 0.0
    batch_count = 0

    for step, batch in enumerate(train_loader):
        ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        labels = batch["cwe_label"].to(DEVICE)

        # Mixed precision forward pass
        with amp.autocast(enabled=use_mixed_precision):
            out1 = model(ids, mask)
            out2 = model(ids, mask)

            loss_scl = scl_criterion(
                torch.cat([out1["features"], out2["features"]], dim=0),
                torch.cat([labels, labels], dim=0),
            )
            loss_rdrop = rdrop_criterion(out1["logits"], out2["logits"], labels, weights=CLASS_WEIGHTS)
            loss = loss_rdrop + scl_weight * loss_scl
            loss = loss / gradient_accumulation_steps  # Scale for accumulation

        # Backward with mixed precision
        scaler.scale(loss).backward()

        total_loss += loss.item()
        total_scl += loss_scl.item() / gradient_accumulation_steps
        total_rdrop += loss_rdrop.item() / gradient_accumulation_steps
        batch_count += 1

        # Accumulate gradients for `gradient_accumulation_steps` batches
        if (step + 1) % gradient_accumulation_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

    n_batches = batch_count
    avg_loss = total_loss / n_batches
    avg_scl = total_scl / n_batches
    avg_rdrop = total_rdrop / n_batches

    # Track losses
    loss_history["loss"].append(avg_loss)
    loss_history["scl"].append(avg_scl)
    loss_history["rdrop"].append(avg_rdrop)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Total: {avg_loss:.4f} | "
        f"R-Drop: {avg_rdrop:.4f} | "
        f"SCL: {avg_scl:.4f}"
    )

    # Save checkpoint after each epoch
    save_checkpoint(epoch, model, optimizer, scheduler, loss_history)

print("\n✅ Training complete! Checkpoint saved for resuming.")